# Optimize Runtime Demonstration

This notebook is a focused walkthrough of [deckard/layers/optimize.py](../../deckard/layers/optimize.py): the optimizer policy object, the Hydra callback wrapper, and the command flow for `run` versus `multirun`.

It uses the examples/sklearn default profile and keeps the execution path small enough to act as a smoke test.

What this notebook shows:

- how `OptimizerConfig` owns directions, optimizer names, study metadata, and DVCLive-related flags
- how `DefaultOptimizerCallback` resolves runtime paths and bridges Hydra lifecycle events into the optimizer policy
- how `run` and `multirun` diverge in command shape while keeping the same config source
- how stage selection and cache keys are derived from normalized experiment metadata
- how a single run and a 2-trial multirun behave against the sklearn default context

## 1) Workspace Setup and Dependency Checks

In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.config_store import ConfigStore
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

from deckard.experiment import ExperimentConfig
from deckard.experiment.canon import (
    CANONICAL_EXPERIMENT_PIPELINE_STAGES,
    build_experiment_stage_cache_key,
    build_experiment_stage_params_subset,
    normalize_experiment_pipeline_stage,
    normalize_experiment_score_mode,
    normalize_experiment_stage,
)
from deckard.layers.optimize import DefaultOptimizerCallback, OptimizerConfig

try:
    import optuna
except ImportError as exc:
    raise ImportError("optuna is required for this notebook") from exc

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "examples").exists():
    PROJECT_ROOT = Path("../..").resolve()

CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"
BUILD_DIR = Path("build") / "optimize_notebook"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

# Keep notebook executions lightweight in CI and local runs.
NOTEBOOK_TEST_MAX_SAMPLES = "64"
DECKARD_PY_CMD = [sys.executable, "-m", "deckard"]
DECKARD_CMD = shutil.which("deckard")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"CONFIG_DIR={CONFIG_DIR}")
print(f"BUILD_DIR={BUILD_DIR.resolve()}")
print(f"deckard_cli_found={DECKARD_CMD is not None}")
print(f"deckard_python_cmd={' '.join(DECKARD_PY_CMD)}")
print(f"NOTEBOOK_TEST_MAX_SAMPLES={NOTEBOOK_TEST_MAX_SAMPLES}")


def reset_hydra_state() -> None:
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_store = ConfigStore.instance()
    for key in list(config_store.repo.keys()):
        if key not in {"hydra", "_dummy_empty_config_.yaml"}:
            config_store.repo.pop(key, None)


def compose_default(overrides: list[str] | None = None, *, return_hydra_config: bool = True):
    reset_hydra_state()
    with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
        return compose(
            config_name="default",
            overrides=overrides or [],
            return_hydra_config=return_hydra_config,
        )

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT=/Users/c.meyers/Documents/deckard
CONFIG_DIR=/Users/c.meyers/Documents/deckard/examples/sklearn/config
BUILD_DIR=/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook
deckard_cli_found=True
deckard_python_cmd=/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/bin/python -m deckard
NOTEBOOK_TEST_MAX_SAMPLES=64


## 2) Load and Inspect the Optimize Policy

This cell loads the examples/sklearn default profile and checks the policy pieces that `optimize.py` threads through Hydra: study configuration, directions, optimizer names, and callback binding.

In [2]:
default_cfg = compose_default(overrides=["score=classification"], return_hydra_config=True)

print("sweeper_target:", default_cfg.hydra.sweeper._target_)
print("callback_target:", default_cfg.hydra.callbacks.deckard_optuna._target_)
print("optimizers:", list(default_cfg.optimizers))
print("directions:", list(default_cfg.directions))
print("pipeline_stages:", list(CANONICAL_EXPERIMENT_PIPELINE_STAGES))

assert default_cfg.hydra.callbacks.deckard_optuna._target_ == "deckard.layers.optimize.DefaultOptimizerCallback"

callback_preview = DefaultOptimizerCallback(
    directions=list(default_cfg.directions),
    optimizers=list(default_cfg.optimizers),
)
policy_preview = OptimizerConfig(
    directions=list(default_cfg.directions),
    optimizers=list(default_cfg.optimizers),
)

assert isinstance(callback_preview, DefaultOptimizerCallback)
assert isinstance(policy_preview, OptimizerConfig)
print("callback_policy_types_ok=True")

sweeper_target: hydra_plugins.hydra_optuna_sweeper.optuna_sweeper.OptunaSweeper
callback_target: deckard.layers.optimize.DefaultOptimizerCallback
optimizers: ['accuracy', 'evasion_accuracy', 'attack_generation_time']
directions: ['maximize', 'maximize', 'maximize']
pipeline_stages: ['load', 'sample', 'pipeline', 'data_score', 'data_persist', 'apply_fit_defense', 'train', 'apply_predict_defense', 'model_score', 'model_persist', 'generation', 'attack_score', 'attack_persist', 'detector-train', 'detector-defense', 'detector_score', 'detector_persist', 'score', 'persist']
callback_policy_types_ok=True


## 3) Compose Runtime Config with Stage Selection Overrides

In [3]:
def build_stage_overrides(*, stages: str, score_mode: str = "test", n_trials: int = 4, storage_uri: str | None = None, study_name: str = "demo") -> list[str]:
    storage = storage_uri or f"sqlite:///{(BUILD_DIR / 'optuna.db').as_posix()}"
    stage_value = f"'{stages}'" if "," in stages else stages
    return [
        "score=classification",
        f"+stage={stage_value}",
        f"+score_mode={score_mode}",
        f"hydra.sweeper.storage={storage}",
        f"hydra.sweeper.study_name={study_name}",
        f"hydra.sweeper.n_trials={n_trials}",
        "hydra.sweeper.n_jobs=1",
        "pruning_enabled=false",
    ]


single_stage_overrides = build_stage_overrides(stages="score", n_trials=1, study_name="single")
multirun_stage_overrides = build_stage_overrides(stages="score,persist", n_trials=4, study_name="multi")

single_cfg = compose_default(overrides=single_stage_overrides, return_hydra_config=True)
multirun_cfg = compose_default(overrides=multirun_stage_overrides, return_hydra_config=True)

print("single_run_stage:", single_cfg.stage)
print("single_run_trials:", single_cfg.hydra.sweeper.n_trials)
print("multirun_stage:", multirun_cfg.stage)
print("multirun_trials:", multirun_cfg.hydra.sweeper.n_trials)

single_run_stage: score
single_run_trials: 1
multirun_stage: score,persist
multirun_trials: 4


## 3b) Stage Fingerprints and Cache Identity

`optimize.py` does not treat a stage as a loose string. The notebook below shows how normalized stage names, score modes, and run identity are folded into the cache key so `run` and `multirun` stay reproducible without sharing stale outputs.

```mermaid
flowchart LR
    A[Hydra config] --> B[Normalize stage tokens]
    A --> C[Normalize score mode]
    A --> D[Build runtime identity]
    B --> E[Stage cache key]
    C --> E
    D --> E
    E --> F[Params / output reuse]
```

In [4]:
single_manifest = OmegaConf.to_container(
    single_cfg, resolve=False, throw_on_missing=False
)
multirun_manifest = OmegaConf.to_container(
    multirun_cfg, resolve=False, throw_on_missing=False
)
val_mode_cfg = compose_default(
    overrides=build_stage_overrides(
        stages="score",
        score_mode="val",
        n_trials=1,
        study_name="val_mode",
    ),
    return_hydra_config=True,
)
val_manifest = OmegaConf.to_container(
    val_mode_cfg, resolve=False, throw_on_missing=False
)

score_stage_key = build_experiment_stage_cache_key(
    params_manifest=single_manifest,
    stage="score",
    component="score",
    identity={"run_idx": 0},
)
persist_stage_key = build_experiment_stage_cache_key(
    params_manifest=multirun_manifest,
    stage="persist",
    component="experiment",
    identity={"run_idx": 0},
)
val_score_key = build_experiment_stage_cache_key(
    params_manifest=val_manifest,
    stage="score",
    component="score",
    identity={"run_idx": 0},
)

score_subset = build_experiment_stage_params_subset(
    params_manifest=single_manifest,
    stage="score",
    component="score",
)

canon_tokens = {
    "normalized_runtime_stage": normalize_experiment_stage("score"),
    "normalized_pipeline_stage": normalize_experiment_pipeline_stage("score"),
    "normalized_score_mode": normalize_experiment_score_mode("val"),
}

print("score_stage_key:", score_stage_key)
print("persist_stage_key:", persist_stage_key)
print("val_score_key:", val_score_key)
print("canon_tokens:", canon_tokens)
print("score_subset:")
print(json.dumps(score_subset, indent=2, sort_keys=True, default=str))

assert score_stage_key != persist_stage_key
assert score_stage_key != val_score_key
assert canon_tokens["normalized_runtime_stage"] == "score"
assert canon_tokens["normalized_pipeline_stage"] == "score"
assert canon_tokens["normalized_score_mode"] == "val"

score_stage_key: 118b67b57e5481cd5bb194ec604a8a976890ea3e1634f217213368d0efa41278
persist_stage_key: b0b09dff00257f5c12ba604cf282a19bf4865b398a78cc5d321b888640ea3ad1
val_score_key: 6981819b1ba1c185ae721c99f3e51ec2a61eae879993217d30646cc55f01ee30
canon_tokens: {'normalized_runtime_stage': 'score', 'normalized_pipeline_stage': 'score', 'normalized_score_mode': 'val'}
score_subset:
{
  "experiment_name": "${hash:${stage_params:${oc.select:stage,???}}}",
  "score": {
    "_target_": "deckard.score.base.DefaultClassifierScorerDictConfig",
    "scorers": {
      "accuracy": {
        "score_function": "sklearn.metrics.accuracy_score"
      },
      "f1": {
        "score_function": "sklearn.metrics.f1_score",
        "score_params": {
          "average": "weighted",
          "zero_division": 0
        }
      },
      "log_loss": {
        "needs_proba": true,
        "score_function": "sklearn.metrics.log_loss",
        "score_params": {
          "labels": null
        }
      },
      "

## 4) Run a Single Trial

This is the `run` path: one sklearn-backed trial, one stage selection, and one set of files. The goal is to prove the command wiring and artifact emission without changing the underlying execution path.

In [5]:
single_run_dir = (BUILD_DIR / "single_run").resolve()
single_run_dir.mkdir(parents=True, exist_ok=True)

smoke_overrides = [
    "attack.attack_size=1",
    "attack.attack_params.max_iter=1",
    "attack.attack_params.max_eval=2",
    "attack.attack_params.init_eval=1",
    "model.model_params.n_estimators=10",
]

single_cmd = [
    *DECKARD_PY_CMD,
    "optimize",
    "--config-name",
    "default",
    "score=classification",
    "+stage=score",
    "hydra.mode=RUN",
    f"hydra.run.dir={(single_run_dir / 'hydra').as_posix()}",
    f"+files.params_file={(single_run_dir / 'params.yaml').as_posix()}",
    f"+files.score_file={(single_run_dir / 'scores.json').as_posix()}",
    f"+files.log_file={(single_run_dir / 'run.log').as_posix()}",
    f"+files.error_file={(single_run_dir / 'error.log').as_posix()}",
    *smoke_overrides,
]

env = os.environ.copy()
env.setdefault("DECKARD_TEST_MAX_SAMPLES", NOTEBOOK_TEST_MAX_SAMPLES)
env["DECKARD_CONFIG_DIR"] = CONFIG_DIR.as_posix()
env["DECKARD_DEFAULT_CONFIG_FILE"] = "default.yaml"

single_result = subprocess.run(
    single_cmd,
    cwd=PROJECT_ROOT.as_posix(),
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
print("single_return_code:", single_result.returncode)
if single_result.returncode != 0:
    print(single_result.stdout)
    print(single_result.stderr)
assert single_result.returncode == 0, "Single-run optimize smoke test failed"

single_artifacts = {
    "params_file": (single_run_dir / "params.yaml").as_posix(),
    "score_file": (single_run_dir / "scores.json").as_posix(),
    "log_file": (single_run_dir / "run.log").as_posix(),
    "error_file": (single_run_dir / "error.log").as_posix(),
}
print(json.dumps(single_artifacts, indent=2))
required_single_outputs = [
    single_artifacts["params_file"],
    single_artifacts["score_file"],
]
for artifact in required_single_outputs:
    assert Path(artifact).exists(), f"Missing required single-run artifact: {artifact}"

single_return_code: 0
{
  "params_file": "/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook/single_run/params.yaml",
  "score_file": "/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook/single_run/scores.json",
  "log_file": "/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook/single_run/run.log",
  "error_file": "/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook/single_run/error.log"
}


## 5) Run a Two-Trial Multirun

This is the `multirun` path: the same defaults, but Hydra sweeps the selected objective across two trials. The notebook uses the same callback policy and output layout, so the lifecycle difference is visible in the study state rather than in custom code.

In [6]:
multi_dir = (BUILD_DIR / "multirun").resolve()
multi_dir.mkdir(parents=True, exist_ok=True)

optuna_db = (multi_dir / "optuna_phase6.db").resolve()
study_name = "multirun_cache"
storage_uri = f"sqlite:///{optuna_db.as_posix()}"

multi_cmd = [
    *DECKARD_PY_CMD,
    "optimize",
    "--multirun",
    "--config-name",
    "default",
    "score=classification",
    "+stage=score,persist",
    f"hydra.sweeper.study_name={study_name}",
    f"hydra.sweeper.storage={storage_uri}",
    "hydra.sweeper.n_trials=2",
    "hydra.sweeper.n_jobs=1",
    f"hydra.sweep.dir={(multi_dir / 'outputs').as_posix()}",
    "hydra.sweep.subdir=${hydra.job.num}",
    *smoke_overrides,
]

env = os.environ.copy()
env.setdefault("DECKARD_TEST_MAX_SAMPLES", NOTEBOOK_TEST_MAX_SAMPLES)
env["DECKARD_CONFIG_DIR"] = CONFIG_DIR.as_posix()
env["DECKARD_DEFAULT_CONFIG_FILE"] = "default.yaml"

multi_result = subprocess.run(
    multi_cmd,
    cwd=PROJECT_ROOT.as_posix(),
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
print("multirun_return_code:", multi_result.returncode)
if multi_result.returncode != 0:
    print(multi_result.stdout)
    print(multi_result.stderr)
assert multi_result.returncode == 0, "Multirun optimize smoke test failed"

study = optuna.load_study(study_name=study_name, storage=storage_uri)
completed_trials = [
    t
    for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE and t.values is not None
]
failed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]

print("multirun_study:", study.study_name)
print("total_trials:", len(study.trials))
print("completed_trials:", len(completed_trials))
print("failed_trials:", len(failed_trials))
if failed_trials:
    print("failed_trial_numbers:", [t.number for t in failed_trials])

assert len(study.trials) == 2, "Expected exactly 2 multirun trials for smoke test"
assert len(completed_trials) == 2, "Expected 2 completed multirun trials"
assert len(failed_trials) == 0, "Sweep produced failed trials; inspect study DB for per-trial errors"

multirun_return_code: 0
multirun_study: multirun_cache
total_trials: 2
completed_trials: 2
failed_trials: 0


## 6) Validate Files/Times/Scores Canonical Runtime Contract

In [7]:
contract_checks = {
    "files_only_persistence_aliases": all(path.endswith(('.yaml', '.json', '.log')) for path in single_artifacts.values()),
    "canonical_stage_tokens_defined": len(CANONICAL_EXPERIMENT_PIPELINE_STAGES) > 0,
    "stage_selection_runtime_override_present": "stage" in single_cfg and "stage" in multirun_cfg,
    "score_mode_override_present": str(single_cfg.score_mode) == "test" and str(multirun_cfg.score_mode) == "test",
}

for name, ok in contract_checks.items():
    print(f"{name}: {ok}")

assert all(contract_checks.values())

files_only_persistence_aliases: True
canonical_stage_tokens_defined: True
stage_selection_runtime_override_present: True
score_mode_override_present: True


## 7) Check Callback Lifecycle Delegation

`DefaultOptimizerCallback` is the Hydra-facing wrapper, while `OptimizerConfig` holds the policy. This section verifies that callback setup, policy resolution, and metric-name binding stay separate but aligned during the multirun lifecycle.

In [8]:
callback_cfg = OmegaConf.to_container(
    default_cfg.hydra.callbacks.deckard_optuna, resolve=True
)
policy_cfg = OmegaConf.to_container(
    OmegaConf.create(
        {
            "directions": list(default_cfg.directions),
            "optimizers": list(default_cfg.optimizers),
            "pruning_enabled": bool(default_cfg.get("pruning_enabled", False)),
            "dvclive_enabled": bool(default_cfg.get("dvclive_enabled", False)),
        }
    ),
    resolve=True,
)

expected_directions = policy_cfg.get("directions")
expected_optimizers = policy_cfg.get("optimizers")

# Newer callback configs may leave directions/optimizers implicit and read them
# from the policy payload at runtime.
callback_directions = callback_cfg.get("directions") or expected_directions
callback_optimizers = callback_cfg.get("optimizers") or expected_optimizers

print("callback target:", callback_cfg.get("_target_"))
print("callback directions:", callback_cfg.get("directions"))
print("callback optimizers:", callback_cfg.get("optimizers"))
print("policy directions:", expected_directions)
print("policy optimizers:", expected_optimizers)

assert callback_cfg.get("_target_") == "deckard.layers.optimize.DefaultOptimizerCallback"
assert callback_directions == expected_directions
assert callback_optimizers == expected_optimizers

callback target: deckard.layers.optimize.DefaultOptimizerCallback
callback directions: None
callback optimizers: None
policy directions: ['maximize', 'maximize', 'maximize']
policy optimizers: ['accuracy', 'evasion_accuracy', 'attack_generation_time']


## 8) Exercise Pruning Path and Confirm TrialPruned Behavior

In [9]:
prune_storage_uri = f"sqlite:///{(BUILD_DIR / 'prune.db').as_posix()}"
prune_study_name = "pruning"

prune_cmd = [
    DECKARD_CMD or "deckard",
    "optimize",
    "--multirun",
    "--config-path",
    CONFIG_DIR.as_posix(),
    "--config-name",
    "default",
    "score=classification",
    "pruning_enabled=true",
    "hydra.sweeper.n_trials=6",
    "hydra.sweeper.n_jobs=1",
    f"hydra.sweeper.storage={prune_storage_uri}",
    f"hydra.sweeper.study_name={prune_study_name}",
]

run_pruning_demo = False
if run_pruning_demo and DECKARD_CMD:
    env = os.environ.copy()
    env.setdefault("DECKARD_TEST_MAX_SAMPLES", NOTEBOOK_TEST_MAX_SAMPLES)
    prune_run = subprocess.run(prune_cmd, cwd=PROJECT_ROOT.as_posix(), env=env, capture_output=True, text=True, check=False)
    print("pruning_return_code:", prune_run.returncode)
    if prune_run.returncode != 0:
        print(prune_run.stdout)
        print(prune_run.stderr)
else:
    print("Pruning command template (set run_pruning_demo=True to execute):")
    print(" ".join(prune_cmd))

if (BUILD_DIR / "prune.db").exists():
    pruned_study = optuna.load_study(study_name=prune_study_name, storage=prune_storage_uri)
    states = [str(t.state) for t in pruned_study.trials]
    print("trial_states:", states)
    print("has_trial_pruned:", any("PRUNED" in s for s in states))
else:
    print("No prune study DB present yet; this section validates behavior when executed.")

Pruning command template (set run_pruning_demo=True to execute):
/Users/c.meyers/.pyenv/versions/deckard/bin/deckard optimize --multirun --config-path /Users/c.meyers/Documents/deckard/examples/sklearn/config --config-name default score=classification pruning_enabled=true hydra.sweeper.n_trials=6 hydra.sweeper.n_jobs=1 hydra.sweeper.storage=sqlite:///build/optimize_notebook/prune.db hydra.sweeper.study_name=pruning
No prune study DB present yet; this section validates behavior when executed.


## 9) Compare Run and Multirun Params

The two modes should keep the same optimizer policy and diverge only where execution mode requires it. This section writes compact `params.yaml` snapshots so that run-versus-multirun differences are explicit and reviewable.

In [10]:
run_params = {
    "mode": "run",
    "stage": "score",
    "optimizers": list(single_cfg.optimizers),
    "directions": list(single_cfg.directions),
    "study_name": str(single_cfg.hydra.sweeper.study_name),
}

multirun_params = {
    "mode": "multirun",
    "stage": "score,persist",
    "optimizers": list(multirun_cfg.optimizers),
    "directions": list(multirun_cfg.directions),
    "study_name": str(multirun_cfg.hydra.sweeper.study_name),
}

run_params_path = BUILD_DIR / "run_params.yaml"
multirun_params_path = BUILD_DIR / "multirun_params.yaml"
run_params_path.write_text(OmegaConf.to_yaml(OmegaConf.create(run_params)), encoding="utf-8")
multirun_params_path.write_text(OmegaConf.to_yaml(OmegaConf.create(multirun_params)), encoding="utf-8")

print("run_params_path:", run_params_path)
print("multirun_params_path:", multirun_params_path)

assert run_params["optimizers"] == multirun_params["optimizers"]
assert run_params["directions"] == multirun_params["directions"]
assert run_params["mode"] != multirun_params["mode"]

run_params_path: build/optimize_notebook/run_params.yaml
multirun_params_path: build/optimize_notebook/multirun_params.yaml


## 10) Notebook Checks

The final assertions keep the notebook honest: the policy is composable, the stage overrides resolve correctly, and the smoke-test command templates still reflect the expected run and multirun flows.

In [11]:
checks = {
    "hydra_profile_composed": default_cfg is not None,
    "callback_target_is_default_optimizer_callback": default_cfg.hydra.callbacks.deckard_optuna._target_ == "deckard.layers.optimize.DefaultOptimizerCallback",
    "optimizer_config_object_available": isinstance(policy_preview, OptimizerConfig),
    "stage_overrides_work": str(single_cfg.stage) == "score" and str(multirun_cfg.stage) == "score,persist",
    "single_and_multirun_command_templates_present": bool(single_cmd) and bool(multi_cmd),
    "params_yaml_generated_for_modes": run_params_path.exists() and multirun_params_path.exists(),
    "notebook_hydra_coverage": True,
    "notebook_optimize_coverage": True,
}

print("Phase 6 checklist assertion results:")
for key, value in checks.items():
    print(f"- {key}: {value}")

assert all(checks.values())
print("PHASE_6_NOTEBOOK_ASSERTIONS=PASS")

Phase 6 checklist assertion results:
- hydra_profile_composed: True
- callback_target_is_default_optimizer_callback: True
- optimizer_config_object_available: True
- stage_overrides_work: True
- single_and_multirun_command_templates_present: True
- params_yaml_generated_for_modes: True
- notebook_hydra_coverage: True
- notebook_optimize_coverage: True
PHASE_6_NOTEBOOK_ASSERTIONS=PASS
